# Uno Feature Regression

Generate training feature values and train a hybrid regression model from feature values to aberration coefficients in one Colab notebook.

Model direction:

```text
feature values -> aberration coefficients
```

Complex quantities use real/imaginary components. The model is a calibrated linear inverse baseline plus a residual MLP.

Scalability note: the training-data generator is isolated in one section. This version materializes a moderate table in memory. Later, the same parameter-case generator can be replaced by a genetic algorithm or sharded sampler, and the resulting feature tables can be streamed into the same feature/target interface.

Results are saved first inside the Colab runtime under `/content/Aberration-Simulation/training_results/feature_regression`. The Google Drive cell is optional and only mirrors that folder when enabled.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone or Pull Latest GitHub Code

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
importlib.invalidate_caches()
print("repo root:", ROOT)
print("commit:", commit)

## 3. Install and Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")
if importlib.util.find_spec("torch") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)

print("Dependencies are ready.")

## 4. Imports

In [ ]:
from pathlib import Path
import csv
import importlib
import shutil
import sys
import zipfile

import numpy as np

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp
from aberration_simulation.line_profiles import extract_line_profiles_from_stack
from aberration_simulation.optics import SimulationConfig, run_simulation, save_npz
from aberration_simulation.uno_conventions import (
    add_complex_columns,
    compute_line_characteristics,
    compute_uno_values,
    select_under_over_pairs,
)

sys.modules.pop("feature_regression_model", None)
importlib.invalidate_caches()
from feature_regression_model import (
    FEATURE_COLUMNS,
    describe_hybrid_model,
    TARGET_COLUMNS,
    train_hybrid_regressor,
)

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())
print("feature_regression_model:", ROOT / "scripts" / "feature_regression_model.py")

## 5. Generate Training Parameter Cases

In [ ]:
OUTPUT_DIR = ROOT / "training_results" / "feature_regression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
C1_OFFSETS = [UNDER_FOCUS_C1_OFFSET, OVER_FOCUS_C1_OFFSET]
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

BASELINE_PARAMETERS = {
    "C1_offset": 0,
    "A3_amp": 0,
    "S3_amp": 0,
    "A2_amp": 0,
    "B2_amp": 0,
    "C1": 0,
    "C3": 0,
    "A1_amp": 0,
    "A1_phase": 0,
    "A2_phase": 0,
    "A3_phase": 0,
    "S3_phase": 0,
    "B2_phase": 0,
}

COMBINATION_FIELDS = (
    "sweep_label",
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)

SCALAR_SWEEP_SPECS = [
    {"label": "C1", "input_field": "C1", "input_values": [-80, -40, -20, 20, 40, 80]},
    {"label": "C3", "input_field": "C3", "input_values": [0.1, 0.2, 0.5, 1.0, 1.5, 2.0]},
]

HARMONIC_SWEEP_SPECS = [
    {"label": "A1", "amp_field": "A1_amp", "phase_field": "A1_phase", "amp_values": [5, 15, 30, 60], "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5]},
    {"label": "B2/C21", "amp_field": "B2_amp", "phase_field": "B2_phase", "amp_values": [0.5, 1.0, 2.0, 3.0], "phase_values": [0, 45, 90, 135, 180, 225, 270, 315]},
    {"label": "A2", "amp_field": "A2_amp", "phase_field": "A2_phase", "amp_values": [1, 4, 8, 16], "phase_values": [0, 30, 60, 90, 120]},
    {"label": "A3", "amp_field": "A3_amp", "phase_field": "A3_phase", "amp_values": [1, 4, 8, 16], "phase_values": [0, 22.5, 45, 67.5, 90]},
    {"label": "S3/C32", "amp_field": "S3_amp", "phase_field": "S3_phase", "amp_values": [1, 4, 8, 16], "phase_values": [0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5, 180]},
]

WIDE_A3_S3_AMP_VALUES = [0, 1, 4, 8, 16, 25, 50, 75, 100]
WIDE_A3_PHASE_VALUES = [0, 22.5, 45, 67.5, 90]
WIDE_S3_PHASE_VALUES = [0, 45, 90, 135, 180]

C1_A1_C3_GRID_LABEL = "C1_A1_C3_grid"
C1_A1_C3_C1_VALUES = [-40, 0, 40]
C1_A1_C3_C3_VALUES = [0, 0.5, 1.5]
C1_A1_C3_A1_AMP_VALUES = [0, 30, 60]
C1_A1_C3_A1_PHASE_VALUES = [0, 45, 90, 135]

A1_B2_S3_GRID_LABEL = "A1_B2_S3_grid"
A1_B2_S3_A1_AMP_VALUES = [0, 60]
A1_B2_S3_A1_PHASE_VALUES = [0, 90]
A1_B2_S3_B2_AMP_VALUES = [0, 3.0]
A1_B2_S3_B2_PHASE_VALUES = [0, 180]
A1_B2_S3_S3_AMP_VALUES = [0, 100]
A1_B2_S3_S3_PHASE_VALUES = [0, 90, 180]

base_cases = []
for spec in SCALAR_SWEEP_SPECS:
    for value in spec["input_values"]:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = spec["label"]
        params[spec["input_field"]] = value
        base_cases.append(params)

for spec in HARMONIC_SWEEP_SPECS:
    for amp in spec["amp_values"]:
        for phase in spec["phase_values"]:
            params = dict(BASELINE_PARAMETERS)
            params["sweep_label"] = spec["label"]
            params[spec["amp_field"]] = amp
            params[spec["phase_field"]] = phase
            base_cases.append(params)

for amp in WIDE_A3_S3_AMP_VALUES:
    for phase in WIDE_A3_PHASE_VALUES:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = "A3_wide"
        params["A3_amp"] = amp
        params["A3_phase"] = phase
        base_cases.append(params)
    for phase in WIDE_S3_PHASE_VALUES:
        params = dict(BASELINE_PARAMETERS)
        params["sweep_label"] = "S3_C32_wide"
        params["S3_amp"] = amp
        params["S3_phase"] = phase
        base_cases.append(params)

for c1_value in C1_A1_C3_C1_VALUES:
    for c3_value in C1_A1_C3_C3_VALUES:
        for a1_amp in C1_A1_C3_A1_AMP_VALUES:
            for a1_phase in C1_A1_C3_A1_PHASE_VALUES:
                params = dict(BASELINE_PARAMETERS)
                params["sweep_label"] = C1_A1_C3_GRID_LABEL
                params["C1"] = c1_value
                params["C3"] = c3_value
                params["A1_amp"] = a1_amp
                params["A1_phase"] = a1_phase
                base_cases.append(params)

for a1_amp in A1_B2_S3_A1_AMP_VALUES:
    for a1_phase in A1_B2_S3_A1_PHASE_VALUES:
        for b2_amp in A1_B2_S3_B2_AMP_VALUES:
            for b2_phase in A1_B2_S3_B2_PHASE_VALUES:
                for s3_amp in A1_B2_S3_S3_AMP_VALUES:
                    for s3_phase in A1_B2_S3_S3_PHASE_VALUES:
                        params = dict(BASELINE_PARAMETERS)
                        params["sweep_label"] = A1_B2_S3_GRID_LABEL
                        params["A1_amp"] = a1_amp
                        params["A1_phase"] = a1_phase
                        params["B2_amp"] = b2_amp
                        params["B2_phase"] = b2_phase
                        params["S3_amp"] = s3_amp
                        params["S3_phase"] = s3_phase
                        base_cases.append(params)

parameters = []
for base_case in base_cases:
    for c1_offset in C1_OFFSETS:
        params = dict(base_case)
        params["C1_offset"] = c1_offset
        parameters.append(params)

print("base cases:", len(base_cases))
print("simulated probe images, including C1 offsets:", len(parameters))
for label in sorted({case["sweep_label"] for case in base_cases}):
    print(f"  {label}: {sum(case['sweep_label'] == label for case in base_cases)} base cases")

## 6. Configure Batched Probe Simulation


In [ ]:
config = SimulationConfig(
    pix_dim=(256, 256),
    real_dim=(1280, 1280),
    app=30,
    sigma=2,
)

# Keep GPU computation vectorized within each batch, but avoid holding all probes
# and FFT temporaries for the full training grid at once.
BATCH_BASE_CASES = 64
BATCH_PROBE_IMAGES = BATCH_BASE_CASES * len(C1_OFFSETS)
ctf_bytes = np.prod(config.pix_dim) * BATCH_PROBE_IMAGES * np.dtype(np.complex128).itemsize
print("batch base cases:", BATCH_BASE_CASES)
print("batch probe images, including C1 offsets:", BATCH_PROBE_IMAGES)
print("approx CTF tensor per batch, GiB:", ctf_bytes / 1024**3)
print("total base cases to process:", len(base_cases))


## 7. Compute Batched Feature Values and Save Training Table


In [ ]:
rows = []
print("expected under/over pairs:", len(base_cases))

for batch_start in range(0, len(base_cases), BATCH_BASE_CASES):
    batch_base_cases = base_cases[batch_start: batch_start + BATCH_BASE_CASES]
    batch_parameters = []
    for base_case in batch_base_cases:
        for c1_offset in C1_OFFSETS:
            params = dict(base_case)
            params["C1_offset"] = c1_offset
            batch_parameters.append(params)

    probe_images, selected = run_simulation(config, batch_parameters)
    simulation_records = [dict(params) for params in batch_parameters]
    batch_pairs = select_under_over_pairs(
        simulation_records,
        COMBINATION_FIELDS,
        under_focus_c1_offset=UNDER_FOCUS_C1_OFFSET,
        over_focus_c1_offset=OVER_FOCUS_C1_OFFSET,
    )

    for local_case_index, (params, under_index, over_index) in enumerate(batch_pairs):
        stack = probe_images[:, :, [under_index, over_index]]
        profiles, coords = extract_line_profiles_from_stack(
            stack,
            num_lines=NUM_PROFILE_LINES,
            radius=PROFILE_RADIUS_PIXELS,
        )
        angles_deg = np.asarray(coords["angles_deg"], dtype=float)
        under_chars = compute_line_characteristics(profiles[:, :, 0], PROFILE_RADIUS_PIXELS)
        over_chars = compute_line_characteristics(profiles[:, :, 1], PROFILE_RADIUS_PIXELS)
        feature_values = compute_uno_values(under_chars, over_chars, angles_deg)

        row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
        row["case_index"] = batch_start + local_case_index
        row["under_index"] = batch_start * len(C1_OFFSETS) + under_index
        row["over_index"] = batch_start * len(C1_OFFSETS) + over_index
        row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
        row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
        for name, value in feature_values.items():
            add_complex_columns(row, name, value)
        rows.append(row)

    del probe_images, selected, batch_parameters, simulation_records, batch_pairs
    xp.cuda.Stream.null.synchronize()
    xp.get_default_memory_pool().free_all_blocks()
    print(f"processed {min(batch_start + BATCH_BASE_CASES, len(base_cases))}/{len(base_cases)} base cases")

CSV_PATH = OUTPUT_DIR / "training_features.csv"
with CSV_PATH.open("w", newline="") as handle:
    fieldnames = list(rows[0].keys())
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("saved training feature table:", CSV_PATH)
print("rows:", len(rows))
for label in sorted({row["sweep_label"] for row in rows}):
    print(f"  {label}: {sum(row['sweep_label'] == label for row in rows)} rows")


## 8. Train Hybrid Feature Regressor

In [ ]:
print("feature columns:")
for name in FEATURE_COLUMNS:
    print(" ", name)
print("target columns:")
for name in TARGET_COLUMNS:
    print(" ", name)

print("model structure:")
print(describe_hybrid_model())

metrics = train_hybrid_regressor(
    CSV_PATH,
    OUTPUT_DIR,
    test_fraction=0.2,
    seed=7,
    epochs=2500,
    learning_rate=2e-3,
    residual_penalty=1e-3,
)
print("training complete")
print("overall test target metrics:")
for name, item in metrics["test_targets"].items():
    print(f"  {name}: MAE={item['mae']:.4g}, RMSE={item['rmse']:.4g}")

## 9. Save and Download Results

In [ ]:
ZIP_PATH = OUTPUT_DIR / "feature_regression_results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path == ZIP_PATH or path.is_dir():
            continue
        zf.write(path, path.name)
print("saved:", ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("Browser download skipped or unsupported:", exc)

## 10. Optional: Mirror Results to Google Drive

In [ ]:
# Set this to True when you want Colab to mount Google Drive and copy results there.
MIRROR_TO_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/Aberration-Simulation/training_results/feature_regression")

if MIRROR_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)
    print("mirrored results to:", DRIVE_OUTPUT_DIR)
else:
    print("Google Drive mirroring disabled. Set MIRROR_TO_GOOGLE_DRIVE=True and rerun this cell to copy results.")